In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer
import torch

In [3]:
from huggingface_hub import login
login()

In [4]:
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M")

In [7]:
print(tokenizer.eos_token)
print(tokenizer.eos_token_id)

<|endoftext|>
0


In [ ]:
data = load_dataset("HuggingFaceFW/fineweb-edu",
                       name="sample-10BT",
                       split="train",
                       streaming=False)



In [ ]:




for exaple in data:
    
    
    print(exaple["text"][:100])
    
        
    break




The Independent Jane
For all the love, romance and scandal in Jane Austen’s books, what they are rea


0.0

In [ ]:
def get_micro_batch():
    
    for text in data.take(1):
        text = text["text"]
        tokens = tokenizer.encode(text)
    tokens = torch.tensor(tokens[:100])
    x = tokens[:-1]
    y = tokens[1:]

    return x.unsqueeze(0) , y.unsqueeze(0)
    
x , y = get_micro_batch()



In [9]:
torch.save((x,y), "Debug_batch.pt")

In [ ]:
import os
import numpy as np
from tqdm.notebook import tqdm
import multiprocessing as mp

nprocs = max(1, os.cpu_count()//2)
SHARD_SIZE = 100_000
OUT_DIR = "tensor_data"

data = load_dataset("HuggingFaceFW/fineweb-edu",
                       name="sample-10BT",
                       split="train" , 
                       streaming=True)

data = data.take(500)

current_tokens = []

os.makedirs("tensor_data", exist_ok=True)

def tokenize(example):
    tokens = tokenizer.encode(example["text"])
    tokens_np = np.array(tokens , dtype=np.uint16)

    return tokens_np

def write_shard(args):
    idx, tokens = args
    path = os.path.join(OUT_DIR, f"shard_{idx:04d}.npy")
    np.save(path, tokens)
    print(f"Saved shard {idx} ({len(tokens):,} tokens)")

current_tokens = np.empty((SHARD_SIZE,), dtype=np.uint16)
token_count = 0
shard_idx = 0
progress = tqdm(total=SHARD_SIZE, desc=f"Shard {shard_idx}", unit="tok")

with mp.Pool(nprocs) as pool:
    for tokens in pool.imap(tokenize,data , chunksize = 16):
        needed = len(tokens)
    
        while needed > 0:
            space = SHARD_SIZE - token_count
            to_write = min(space , needed)
            offset = len(tokens) - needed

            current_tokens[token_count:token_count + to_write] = tokens[offset:offset + to_write]
            token_count += to_write
            needed -= to_write
            progress.update(to_write)

            if token_count == SHARD_SIZE:
                np.save(os.path.join(OUT_DIR, f"shard_{shard_idx:04d}.npy"), current_tokens)
                print(f"\nSaved shard {shard_idx}")
                shard_idx += 1
                token_count = 0
                progress = tqdm(total=SHARD_SIZE, desc=f"Shard {shard_idx}", unit="tok")

    
    if token_count > 0:
        np.save(os.path.join(OUT_DIR, f"shard_{shard_idx:04d}.npy"), current_tokens[:token_count])
        print(f"Saved final shard {shard_idx} ({token_count:,} tokens)")


    